In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), "ERROR: T4 GPU is not enabled! Go to Runtime -> Change runtime type -> T4 GPU."
print(f"GPU ready: {torch.cuda.get_device_name(0)}")

zip_file = Path("e2e_eval_colab.zip")
assert zip_file.exists() and zipfile.is_zipfile(zip_file), "ERROR: e2e_eval_colab.zip is not fully uploaded!"

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "chromadb>=1.0.0", "rank-bm25", "pyyaml",
        "langchain-community", "transformers", "accelerate", "sentence-transformers",
    ],
    check=True,
)

for folder in (
    "Query Understanding and Routing",
    "Retrieval",
    "Response Synthesis",
    "Document Ingestion and Parsing",
    "Evaluation",
):
    subprocess.run(["rm", "-rf", folder], check=True)
Path("path.py").unlink(missing_ok=True)
Path("e2e_eval_results.zip").unlink(missing_ok=True)

subprocess.run(["unzip", "-q", "-o", str(zip_file)], check=True)

root = Path.cwd()
extra_paths = [
    str(root / "Evaluation" / "End-to-end RAG Evals"),
    str(root / "Response Synthesis"),
    str(root / "Retrieval"),
    str(root / "Query Understanding and Routing"),
    str(root / "Document Ingestion and Parsing" / "Metadata"),
    str(root / "Document Ingestion and Parsing" / "Provenance and Traceability"),
    str(root),
]
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
prev_pp = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = os.pathsep.join(extra_paths + ([prev_pp] if prev_pp else []))

eval_script = root / "Evaluation" / "End-to-end RAG Evals" / "eval_e2e.py"
proc = subprocess.Popen(
    [sys.executable, "-u", str(eval_script)],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")

if proc.wait() != 0:
    raise RuntimeError("eval_e2e.py encountered an error — see detailed logs above.")

subprocess.run(
    [
        "zip", "-q", "-j", "e2e_eval_results.zip",
        "Evaluation/End-to-end RAG Evals/reports/generation_report.json",
        "Evaluation/End-to-end RAG Evals/reports/task_report.json",
        "Evaluation/End-to-end RAG Evals/reports/performance_report.json",
        "Evaluation/End-to-end RAG Evals/reports/final_evaluation_report.json",
    ],
    check=True,
)
from google.colab import files
files.download("e2e_eval_results.zip")

GPU sẵn sàng: Tesla T4
>>> Bắt đầu End-to-End & System Performance Evaluation (150 mẫu) trên: CUDA
>>> Đang nạp mô hình sinh văn bản 'Qwen/Qwen3-4B' lên cuda...

Fetching 3 files: 100%|██████████| 3/3 [01:48<00:00, 36.25s/it]

Loading weights: 100%|██████████| 398/398 [00:33<00:00, 11.90it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 17409.85it/s]
>>> Đang chạy 1 lượt Warm-up mô hình...

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3670.81it/s]

>>> [PHA 1] Thực thi toàn trình 150 mẫu và ghi nhận System Performance...
  ... Đã thực thi 10/150 mẫu (41.8s)
Requested version(s) ['5.0.0'] not found in catalog
Task 'task_001': requested version not in knowledge base, abstaining
Requested version(s) ['5.0.0'] not found in catalog
Task 'task_001': requested version not in knowledge base, abstaining
Abstention triggered cho query='In application security verification standards, what is the difference between a major release and a minor release regarding requirement nu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>